# Memoria Procedimental en Agentes de IA

## ¿Qué es la Memoria Procedimental?

La **memoria procedimental** en agentes de IA hace referencia a la capacidad del agente de **recordar cómo hacer las cosas**, es decir, almacenar instrucciones o "prompts" que guían su comportamiento.

A diferencia de la memoria episódica (que recuerda eventos pasados) o la memoria semántica (que recuerda hechos), la memoria procedimental define el **procedimiento** que sigue el agente para realizar tareas.

En este notebook veremos:
1. Cómo un agente carga instrucciones desde un almacén (store)
2. Cómo optimizar automáticamente esas instrucciones con feedback del usuario
3. Cómo aplicar esto en un sistema multi-agente con supervisor

---
## Paso 1: Instalación de dependencias

Instalamos las bibliotecas necesarias:
- **`langmem`**: Biblioteca de LangChain para gestión de memoria en agentes. Incluye herramientas para optimizar prompts automáticamente.
- **`langgraph`**: Framework para construir agentes y flujos de trabajo con grafos.

In [ ]:
%%capture cap --no-stderr
%pip install -U langmem langgraph langchain-openai python-dotenv

---
## Paso 2: Configurar la API Key de OpenAI

El archivo `.env` debe estar en la misma carpeta que este notebook (`Clase 2.2/`) y contener:
```
OPENAI_API_KEY=sk-proj-...
```
> **Nunca compartas tu API key en el chat ni en el código.**

In [14]:
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(dotenv_path=Path().resolve() / ".env", override=True)

# Desactivar tracing y limpiar variables que redirigen a proxies
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ.pop("LANGSMITH_API_KEY", None)
os.environ.pop("OPENAI_BASE_URL", None)
os.environ.pop("OPENAI_API_BASE", None)
os.environ.pop("OPENAI_PROXY", None)

# Diagnóstico
key = os.environ.get("OPENAI_API_KEY", "")
base_url = os.environ.get("OPENAI_BASE_URL", "none")
print(f"OPENAI_API_KEY : {key[:8]}... (longitud: {len(key)})")
print(f"OPENAI_BASE_URL: {base_url}")
print(f"LANGCHAIN_TRACING_V2: {os.environ.get('LANGCHAIN_TRACING_V2')}")

OPENAI_API_KEY : sk-proj-... (longitud: 164)
OPENAI_BASE_URL: none
LANGCHAIN_TRACING_V2: false


---
## Paso 2: Crear el almacén de instrucciones (InMemoryStore)

`InMemoryStore` es un almacén de datos **en memoria** (se pierde al reiniciar el programa). Funciona como una base de datos temporal tipo clave-valor.

- `store.put(namespace, key, value)`: Guarda un valor en el almacén.
  - **namespace**: Tupla que agrupa elementos relacionados, como una carpeta. Aquí usamos `("instructions",)`.
  - **key**: Nombre único del elemento dentro del namespace.
  - **value**: El dato a guardar (en este caso, un diccionario con el prompt inicial).

El prompt inicial es simple: `"Write good emails."` (Escribe buenos correos).

In [15]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("instructions",), key="agent_instructions", value={"prompt": "Write good emails."})

---
## Paso 3: Crear el agente con memoria procedimental

Aquí definimos tres componentes clave:

### 3.1 Herramienta `draft_email`
Una función que simula guardar un borrador de correo. El agente la usará como herramienta disponible. En un sistema real, aquí se conectaría con Gmail u otro proveedor.

### 3.2 Función `prompt`
Esta función se ejecuta **antes de cada llamada al LLM** y:
1. Lee las instrucciones actuales desde el `store`
2. Las inyecta como mensaje de sistema al inicio de la conversación

Esto es la esencia de la memoria procedimental: **el agente consulta sus instrucciones guardadas en cada interacción**.

### 3.3 `create_react_agent`
Crea un agente con arquitectura **ReAct** (Reasoning + Acting): el agente razona sobre qué herramienta usar y luego actúa.

In [16]:
from langgraph.prebuilt import create_react_agent

def draft_email(to: str, subject: str, body: str):
    """Submit an email draft."""
    return "Draft saved successfully."

def prompt(state):
    item = store.get(("instructions",), key="agent_instructions")
    instructions = item.value["prompt"]
    sys_prompt = {"role": "system", "content": f"## Instructions\n\n{instructions}"}
    return [sys_prompt] + state['messages']

agent = create_react_agent(
    "openai:gpt-4o-mini",
    prompt=prompt,
    tools=[draft_email],
    store=store
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17364\1803668764.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


---
## Paso 4: Primera invocación del agente

Invocamos al agente con una solicitud de correo. El agente:
1. Lee su prompt del store (`"Write good emails."`)
2. Razona sobre cómo escribir el correo
3. Llama a la herramienta `draft_email` con los parámetros correctos

El resultado es básico porque el prompt inicial es muy genérico. No firma con ningún nombre específico ni ofrece opciones de videollamada.

In [17]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]}
)
result['messages'][1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  draft_email (call_WKNj3wP1v7YAEZKC7J5lR1OW)
 Call ID: call_WKNj3wP1v7YAEZKC7J5lR1OW
  Args:
    to: joe@langchain.dev
    subject: Follow-Up Meeting Scheduling
    body: Hi Joe,

I hope this message finds you well. We would like to schedule a follow-up meeting with you on Thursday at noon. Please let us know if this time works for you or if there is another time that’s more convenient.

Looking forward to your reply.

Best regards,

[Your Name]


---
## Paso 5: Optimizar el prompt automáticamente

### ¿Qué es `create_prompt_optimizer`?

`create_prompt_optimizer` es una función de **langmem** que crea un optimizador de prompts. Este optimizador:
- Analiza la trayectoria de conversación (lo que el agente hizo)
- Considera el feedback del usuario
- Reescribe el prompt para mejorar el comportamiento del agente

Es decir, **actualiza automáticamente las instrucciones del agente** basándose en correcciones humanas. Esto es aprendizaje procedimental.

In [18]:
from langmem import create_prompt_optimizer

optimizer = create_prompt_optimizer("openai:gpt-4o-mini")

---
## Paso 6: Aplicar el feedback al optimizador

Aquí ocurre la magia del aprendizaje procedimental:

1. **Obtenemos el prompt actual** del store
2. **Definimos el feedback** del usuario: en este caso, queremos que:
   - El correo siempre se firme como 'William'
   - En solicitudes de reunión, se ofrezca Zoom o Google Meet
3. **Invocamos el optimizador** con:
   - `prompt`: El prompt actual
   - `trajectories`: Lista de tuplas `(mensajes, feedback)` — la conversación pasada + la corrección

El optimizador analiza qué hizo mal el agente y reescribe el prompt para que no vuelva a ocurrir.

In [19]:
current_prompt = store.get(("instructions",), key="agent_instructions").value["prompt"]
feedback = {"request": "Always sign off from 'William'; for meeting requests, offer to schedule on Zoom or Google Meet"}

optimizer_result = optimizer.invoke({
    "prompt": current_prompt,
    "trajectories": [(result["messages"], feedback)]
})

---
## Paso 7: Ver el prompt optimizado

El optimizador generó un nuevo prompt mucho más detallado. Ahora incluye:
- Firma con 'William'
- Instrucciones específicas para reuniones (Zoom/Google Meet)
- Tono profesional
- Ejemplo de correo tipo

El optimizador convirtió una corrección humana vaga en instrucciones claras y accionables.

In [20]:
print(optimizer_result)

Write good emails with a specific sign-off using the name "William" and include suggestions for scheduling on Zoom or Google Meet for any meeting requests.


---
## Paso 8: Guardar el nuevo prompt en el store

Actualizamos el store con el prompt mejorado. La próxima vez que el agente sea invocado, leerá este nuevo prompt desde el store y seguirá las instrucciones actualizadas.

Esto es como **actualizar el manual de procedimientos** del agente.

In [21]:
store.put(("instructions",), key="agent_instructions", value={"prompt": optimizer_result})

---
## Paso 9: Segunda invocación — Verificar la mejora

Invocamos al agente con **exactamente la misma solicitud** que antes. Ahora el agente debería:
- Ofrecer Zoom o Google Meet como opciones de reunión
- Firmar el correo como 'William'

El agente aprendió de la corrección humana sin necesidad de reentrenamiento del modelo.

In [22]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]}
)
result['messages'][1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  draft_email (call_HCExC9Gbb5xYsZQK7JrIaXwD)
 Call ID: call_HCExC9Gbb5xYsZQK7JrIaXwD
  Args:
    to: joe@langchain.dev
    subject: Follow-Up Meeting Scheduling
    body: Hi Joe,

I hope this message finds you well. We would like to schedule a follow-up meeting for this Thursday at noon. Please let me know if this time works for you, or if there's another that you'd prefer.

Looking forward to your reply.

Best regards,
William


---
## Paso 10: Probar con un caso diferente

Ahora probamos con una solicitud de correo diferente (no es una reunión, sino una actualización sobre un release). El agente debería:
- Mantener la firma 'William' (porque está en el prompt)
- No mencionar Zoom/Google Meet (porque no es una reunión)

Esto demuestra que el agente aplica las instrucciones de forma **contextual e inteligente**.

In [23]:
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Let roger@langchain.dev know that the release should be later by 4:00 PM."}]}
)
result['messages'][1].pretty_print()

================================== Ai Message ==================================
Tool Calls:
  draft_email (call_T7xdkqvjX5edfcQYhoKI034r)
 Call ID: call_T7xdkqvjX5edfcQYhoKI034r
  Args:
    to: roger@langchain.dev
    subject: Release Timing Update
    body: Hi Roger,

I wanted to inform you that the release should be scheduled for later, specifically by 4:00 PM. Please let me know if you need any further information or adjustments.

Looking forward to your confirmation.

Best,
William


---
# Sistema Multi-Agente con Memoria Procedimental

Hasta ahora teníamos un solo agente. Ahora escalaremos a un **sistema multi-agente** donde:
- Un **supervisor** decide qué agente usar
- Un **agente de email** maneja correos
- Un **agente de redes sociales** maneja tweets

Cada agente tiene su **propio prompt** guardado en el store, y se pueden optimizar de forma independiente.

---
## Paso 11: Instalar langgraph-supervisor

`langgraph-supervisor` es una extensión que facilita crear un agente supervisor que orquesta a otros agentes.

In [24]:
%%capture cap --no-stderr
%pip install -U langgraph-supervisor

---
## Paso 12: Crear el store y los dos agentes

Creamos un nuevo store con instrucciones separadas para cada agente:
- `email_agent`: Instrucciones para escribir correos
- `twitter_agent`: Instrucciones para escribir tweets

Cada agente tiene su propia función `prompt_*` que lee **solo sus instrucciones** del store. Esto permite actualizarlos de forma independiente.

**Importante:** Cada agente tiene un `name` único (`email_assistant`, `social_media_agent`) para que el supervisor pueda identificarlos y delegarles tareas.

In [25]:
from langgraph.store.memory import InMemoryStore
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

store = InMemoryStore()

store.put(("instructions",), key="email_agent", value={"prompt": "Write good emails. Repeat your draft content to the user after submitting."})
store.put(("instructions",), key="twitter_agent", value={"prompt": "Write fire tweets. Repeat the tweet content to the user upon submission."})

agent_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def draft_email(to: str, subject: str, body: str):
    """Submit an email draft."""
    return "Draft saved successfully."

def prompt_email(state):
    item = store.get(("instructions",), key="email_agent")
    instructions = item.value["prompt"]
    sys_prompt = {"role": "system", "content": f"## Instructions\n\n{instructions}"}
    return [sys_prompt] + state['messages']

email_agent = create_react_agent(
    agent_model,
    prompt=prompt_email,
    tools=[draft_email],
    store=store,
    name="email_assistant",
)

def tweet(to: str, subject: str, body: str):
    """Post a tweet."""
    return "Legendary."

def prompt_social_media(state):
    item = store.get(("instructions",), key="twitter_agent")
    instructions = item.value["prompt"]
    sys_prompt = {"role": "system", "content": f"## Instructions\n\n{instructions}"}
    return [sys_prompt] + state['messages']

social_media_agent = create_react_agent(
    agent_model,
    prompt=prompt_social_media,
    tools=[tweet],
    store=store,
    name="social_media_agent",
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17364\3426572103.py:22: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  email_agent = create_react_agent(
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17364\3426572103.py:40: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  social_media_agent = create_react_agent(


---
## Paso 13: Crear el supervisor

`create_supervisor` crea un agente que actúa como **coordinador**:
- Recibe el mensaje del usuario
- Decide qué agente especialista debe manejarlo
- Delega la tarea al agente correcto
- Retorna la respuesta al usuario

El supervisor también usa un LLM para tomar estas decisiones de enrutamiento.

`workflow.compile(store=store)` compila el grafo completo en una aplicación ejecutable, pasando el store compartido.

In [26]:
from langgraph_supervisor import create_supervisor
from langchain_openai import ChatOpenAI

supervisor_model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

workflow = create_supervisor(
    [email_agent, social_media_agent],
    model=supervisor_model,
    prompt=(
        "You are a team supervisor managing email and tweet assistants to help with correspondence."
    )
)

app = workflow.compile(store=store)

---
## Paso 14: Primera invocación del sistema multi-agente

El supervisor recibe la solicitud y decide que el `email_assistant` debe manejarla.

Accedemos a `result["messages"][3]` porque los mensajes incluyen:
- `[0]`: Mensaje del usuario
- `[1]`: Decisión del supervisor (delegar al email_assistant)
- `[2]`: Llamada a herramienta (draft_email)
- `[3]`: Respuesta final del email_assistant al usuario

In [27]:
result = app.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]},
)

result["messages"][3].pretty_print()

================================== Ai Message ==================================
Name: email_assistant

I have drafted the email to Joe at joe@langchain.dev. Here’s the content:

---

**Subject:** Follow-up Meeting Scheduling

Hi Joe,

I hope this message finds you well. We would like to schedule a follow-up meeting for Thursday at noon. Please let us know if this time works for you or if there are any other times that you would prefer.

Looking forward to your response.

Best regards,

[Your Name]

--- 

Let me know if you need any changes!


---
## Paso 15: Optimizador multi-prompt

`create_multi_prompt_optimizer` es la versión avanzada del optimizador que puede:
- Manejar **múltiples prompts simultáneamente** (uno por agente)
- Aplicar el feedback **solo al agente relevante**
- Mantener sin cambios los prompts de agentes que no necesitan mejora

Esto es crucial en sistemas multi-agente: no queremos que el feedback sobre correos afecte las instrucciones del agente de tweets.

In [28]:
from langmem import create_multi_prompt_optimizer

feedback = {"request": "Always sign off emails from 'William'; for meeting requests, offer to schedule on Zoom or Google Meet"}

optimizer = create_multi_prompt_optimizer("openai:gpt-4o-mini")

---
## Paso 16: La clase `Prompt` de langmem

`Prompt` es un `TypedDict` (diccionario con tipos definidos) que estructura la información de cada prompt para el optimizador:

| Campo | Obligatorio | Descripción |
|-------|-------------|-------------|
| `name` | Sí | Identificador único del prompt |
| `prompt` | Sí | El texto actual del prompt |
| `update_instructions` | No | Guías para modificar el prompt |
| `when_to_update` | No | Condición para actualizar este prompt |

El campo `when_to_update` es especialmente útil en sistemas multi-agente: le dice al optimizador **cuándo** debe o no debe modificar un prompt específico.

In [29]:
from langmem import Prompt
?Prompt

Init signature: Prompt(self, /, *args, **kwargs)
Docstring:     
TypedDict for structured prompt management and optimization.

Example:
    ```python
    from langmem import Prompt

    prompt = Prompt(
        name="extract_entities",
        prompt="Extract key entities from the text:",
        update_instructions="Make minimal changes, only address where"
        " errors have occurred after reasoning over why they occur.",
        when_to_update="If there seem to be errors in recall of named entities.",
    )
    ```

The name and prompt fields are required. Optional fields control optimization:
- update_instructions: Guidelines for modifying the prompt
- when_to_update: Dependencies between prompts during optimization

Use in the prompt optimizers.
File:           c:\users\lenovo\anaconda3\envs\rag_env\lib\site-packages\langmem\prompts\types.py
Type:           _TypedDictMeta
Subclasses:     

---
## Paso 17: Estructurar y optimizar los prompts de los agentes

Preparamos los prompts de ambos agentes como diccionarios con la estructura `Prompt`:

- **`tweet_prompt`**: Solo se actualiza si la generación de tweets necesita mejora.
- **`email_prompt`**: Solo se actualiza si el feedback indica que el email necesita mejorar.

El optimizador analizará la trayectoria de la conversación y el feedback para decidir:
- ¿Cuál prompt necesita cambios?
- ¿Qué cambios específicos aplicar?

En este caso, el feedback es sobre correos → solo debería cambiar `email_prompt`.

In [30]:
# Obtener los prompts actuales del store
email_prompt = store.get(("instructions",), key="email_agent").value['prompt']
tweet_prompt = store.get(("instructions",), key="twitter_agent").value['prompt']

# Estructurar como objetos Prompt con condiciones de actualización
email_prompt = {
    "name": "email_prompt",
    "prompt": email_prompt,
    "when_to_update": "Only if feedback is provided indicating email writing performance needs improved."
}
tweet_prompt = {
    "name": "tweet_prompt",
    "prompt": tweet_prompt,
    "when_to_update": "Only if tweet writing generation needs improvement."
}

# Invocar el optimizador multi-prompt
optimizer_result = optimizer.invoke({
    "prompts": [tweet_prompt, email_prompt],
    "trajectories": [(result["messages"], feedback)]
})

---
## Paso 18: Ver el resultado de la optimización multi-prompt

El resultado muestra los dos prompts:
- **`tweet_prompt`**: Sin cambios (el feedback no mencionó tweets)
- **`email_prompt`**: Actualizado con instrucciones detalladas sobre firma y opciones de videollamada

Esto demuestra la inteligencia del optimizador: aplica los cambios **solo donde son necesarios**.

In [31]:
optimizer_result

[{'name': 'tweet_prompt',
  'prompt': 'Write fire tweets. Repeat the tweet content to the user upon submission.',
  'when_to_update': 'Only if tweet writing generation needs improvement.'},
 {'name': 'email_prompt',
  'prompt': "Write good emails. Always sign off using the user’s preferred name, 'William', and offer to schedule meetings using popular videoconferencing tools like Zoom or Google Meet. Repeat your draft content to the user after submitting.",
  'when_to_update': 'Only if feedback is provided indicating email writing performance needs improved.'}]

---
## Paso 19: Actualizar el store del agente de email

Guardamos el prompt mejorado del agente de email en el store. Usamos `optimizer_result[1]` porque el resultado es una lista donde:
- `[0]` = tweet_prompt (sin cambios)
- `[1]` = email_prompt (actualizado)

El agente de redes sociales no necesita actualización.

In [32]:
store.put(("instructions",), key="email_agent", value={"prompt": optimizer_result[1]['prompt']})

---
## Paso 20: Invocación final — Verificar la mejora en el sistema multi-agente

Probamos el sistema completo con la misma solicitud. Ahora el email_assistant debería:
- Ofrecer Zoom o Google Meet como opciones
- Firmar como 'William'

El supervisor sigue funcionando igual, pero el agente que delega ahora tiene mejores instrucciones.

In [33]:
result = app.invoke(
    {"messages": [
        {"role": "user", "content": "Draft an email to joe@langchain.dev saying that we want to schedule a followup meeting for thursday at noon."}]},
)

result["messages"][3].pretty_print()

================================== Ai Message ==================================
Name: email_assistant

I have drafted the email to Joe regarding scheduling a follow-up meeting for Thursday at noon. Here’s the content:

---

Subject: Follow-up Meeting Scheduling

Hi Joe,

I hope this message finds you well. We would like to schedule a follow-up meeting for Thursday at noon. Please let us know if that works for you or if there’s a better time.

Looking forward to your response.

Best regards,

William

---

If you would like to schedule a meeting using Zoom or Google Meet, please let me know!


In [46]:
print(result['messages'][3].content)

I have drafted the email to Joe regarding scheduling a follow-up meeting for Thursday at noon. Here’s the content:

---

Subject: Follow-up Meeting Scheduling

Hi Joe,

I hope this message finds you well. We would like to schedule a follow-up meeting for Thursday at noon. Please let us know if that works for you or if there’s a better time.

Looking forward to your response.

Best regards,

William

---

If you would like to schedule a meeting using Zoom or Google Meet, please let me know!


---
# Resumen: ¿Qué aprendimos?

## Conceptos clave

| Concepto | Descripción |
|----------|-------------|
| **Memoria Procedimental** | El agente almacena y consulta instrucciones sobre *cómo* realizar tareas |
| **InMemoryStore** | Almacén temporal clave-valor para guardar prompts e instrucciones |
| **Prompt Injection dinámica** | La función `prompt()` inyecta las instrucciones del store antes de cada llamada al LLM |
| **Optimización de Prompts** | `create_prompt_optimizer` actualiza automáticamente las instrucciones basándose en feedback humano |
| **Multi-Prompt Optimizer** | `create_multi_prompt_optimizer` gestiona múltiples prompts y actualiza solo los relevantes |
| **Supervisor Pattern** | Un agente coordinador delega tareas a agentes especialistas |

## Flujo completo

```
Usuario → Supervisor → Agente especialista → Lee prompt del store → LLM → Respuesta
                                                                         ↓
                                               Feedback → Optimizador → Store actualizado
```

## ¿Por qué es importante?

La memoria procedimental permite que los agentes **mejoren con el tiempo** sin reentrenar el modelo base. Es una forma eficiente y escalable de personalizar el comportamiento de agentes de IA en aplicaciones reales.